# 화구(Fireball) 관측 데이터 분석 — 위성 관측은 편향이 없는가?

**데이터 출처**: NASA/JPL Center for Near-Earth Object Studies (CNEOS), Fireball and Bolide Data API
(https://ssd-api.jpl.nasa.gov/fireball.api) — 미국 정부 위성이 1988년부터 관측한 대기권 진입 화구 사건 공식 기록.

**근거 논문**
- Brown, P. et al. (2002). *The flux of small near-Earth objects colliding with the Earth.* Nature, 420, 294–296.
  → 이 위성 관측망이 지구 표면의 약 80%를 감시하는 "near-global coverage"를 갖는다는 것을 확립한 논문.
- Devillepoix, H.A.R. et al. (2020). *A Global Fireball Observatory.* Planetary and Space Science, 191, 105036. DOI: 10.1016/j.pss.2020.105036
  → 대조군 성격의 논문. 전 세계 지상 카메라 관측망(Global Fireball Observatory)을 다 합쳐도 2019년 기준 지구 표면의 **0.6%**만 감시 가능하다고 보고. 위성(80%)과 지상 카메라(0.6%)의 감시 범위 격차를 정량적으로 보여주는 근거.
- Peña-Asensio, E., Socas-Navarro, H., & Seligman, D.Z. (2025). *Error dependencies in the space-based CNEOS fireball database.* Astronomy & Astrophysics. (arXiv:2508.01454)
  → 이 데이터셋의 신뢰도를 정밀 검증한 최신 논문. 2018년 이후 관측 정확도가 크게 개선되었으며, 그 이전 데이터나 일부 고속 이벤트는 오차가 클 수 있음을 지적.

**분석 목적**: 앞서(서랍1 노트북) 확인한 "운석 발견 위치가 남극에 68.7% 편중된다"는 결과가
실제 낙하 분포 때문인지, 지구상의 발견 방식(수색 활동) 때문인지 검증하기 위해,
발견이 아니라 위성이 '관측'한 별도의 공식 데이터로 교차 검증한다.

## 1. 라이브러리 및 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns

print("준비 완료")

아래 칸 실행 → "파일 선택" → `fireball_landings.csv` 업로드

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('fireball_landings.csv')
print("전체 화구 이벤트:", len(df))
df.head()

## 2. 데이터 전처리

`lat`/`lon`은 절댓값으로만 기록되어 있고, 남/서쪽 방향은 `lat-dir`/`lon-dir` 컬럼(N/S/E/W)에 별도로 표기되어 있다.
분석을 위해 남위·서경을 음수로 변환한 부호 있는 좌표(`lat_signed`, `lon_signed`)를 새로 만든다.

In [ ]:
def to_signed(val, direction, neg_dirs):
    if pd.isna(val) or pd.isna(direction):
        return np.nan
    return -val if direction in neg_dirs else val

df['lat_signed'] = df.apply(lambda r: to_signed(r['lat'], r['lat-dir'], ['S']), axis=1)
df['lon_signed'] = df.apply(lambda r: to_signed(r['lon'], r['lon-dir'], ['W']), axis=1)

print("위치 정보가 있는 이벤트:", df.dropna(subset=['lat_signed','lon_signed']).shape[0], "/", len(df))

## 3. 분석 1 — 남극권 관측 비율 (핵심 비교)

앞선 운석 착지(발견) 데이터에서는 남극권(위도 -60도 이하) 비율이 68.7%였다.
같은 기준으로 화구 관측 데이터를 계산해 비교한다.

In [ ]:
geo = df.dropna(subset=['lat_signed','lon_signed'])

antarctica_count = (geo['lat_signed'] < -60).sum()
antarctica_ratio = antarctica_count / len(geo) * 100

print(f"위치 정보 보유 화구 이벤트: {len(geo)}건")
print(f"남극권(위도 -60 이하) 화구 이벤트: {antarctica_count}건 ({antarctica_ratio:.2f}%)")
print()
print("비교:")
print(f"  운석 착지(발견) 데이터 남극 비율: 68.7%")
print(f"  화구(위성 관측) 데이터 남극 비율: {antarctica_ratio:.2f}%")

### 3-1. 시각화

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(['운석 착지\n(발견 기록)', '화구\n(위성 관측)'], [68.7, antarctica_ratio], color=['coral','steelblue'])
plt.ylabel('남극권 비율(%)')
plt.title('발견 방식에 따른 남극권 비율 비교')
plt.show()

### 3-2. 결과 해석

- 운석 착지 데이터에서는 남극권 비율이 68.7%였으나, 화구 관측 데이터에서는 6.57%에 불과하다.
- 두 데이터는 같은 현상(지구에 도달하는 소천체)을 다른 방식으로 기록한다: 화구 데이터는 인공위성이 전 지구를 대상으로 자동 관측한 기록이고, 운석 착지 데이터는 사람이 실제로 찾아서 등재한 기록이다.
- 이 차이는 실제 낙하 분포가 남극에 편중되어 있어서가 아니라, 남극에서의 조직적 탐사(6~7절 참고)가 발견 기록을 편중시켰을 가능성을 강하게 시사한다.

## 4. 분석 2 — 에너지 지표 간 상관관계

`energy`(광학 복사 에너지)와 `impact-e`(추정 충돌 에너지)의 상관관계를 확인한다.

In [ ]:
corr = df[['energy','impact-e']].corr().iloc[0,1]
print(f"energy - impact-e 상관계수: {corr:.4f}")

plt.figure(figsize=(6,5))
plt.scatter(np.log10(df['energy']), np.log10(df['impact-e']), s=10, alpha=0.5)
plt.xlabel('log10(energy)')
plt.ylabel('log10(impact-e)')
plt.title('에너지 지표 간 상관관계 (로그 스케일)')
plt.show()

**해석**: 상관계수가 0.998로 거의 완전한 상관관계를 보인다. 이는 두 지표가 물리적으로 서로 연동되어 산출되는 값임을 확인해준다.

## 5. 분석 3 — 남반구 vs 북반구 에너지 차이 검정

앞선 서랍1 노트북의 Fell/Found 검정과 동일한 절차(정규성 검정 → 적절한 검정 선택)를 적용한다.

In [ ]:
north = geo.loc[geo['lat_signed']>=0, 'energy'].dropna()
south = geo.loc[geo['lat_signed']<0, 'energy'].dropna()
print("북반구 n =", len(north), " 남반구 n =", len(south))

log_n = np.log10(north)
log_s = np.log10(south)

sw_n = stats.shapiro(log_n.sample(min(500,len(log_n)), random_state=42))
sw_s = stats.shapiro(log_s.sample(min(500,len(log_s)), random_state=42))
print("Shapiro-Wilk(북반구):", sw_n)
print("Shapiro-Wilk(남반구):", sw_s)

In [ ]:
# 정규분포가 아니므로 Mann-Whitney U 검정 사용
u_stat, p_val = stats.mannwhitneyu(north, south, alternative='two-sided')
print(f"Mann-Whitney U 검정: U={u_stat:.1f}, p={p_val:.4f}")
print("북반구 중앙값(J):", north.median())
print("남반구 중앙값(J):", south.median())

**해석**: p=0.986으로, 남반구와 북반구 사이 에너지 분포에 통계적으로 유의한 차이가 없다.
이는 위성 기반 관측이 지리적으로 편향되지 않는다는 것을 뒷받침하는 결과이며, 3절의 해석과 일관된다.

## 6. 분석 4 — 회귀분석 (에너지 지표 간 관계식)

`energy`로 `impact-e`를 예측하는 로그-로그 선형회귀를 수행한다.

In [ ]:
reg_df = df.dropna(subset=['energy','impact-e'])
X = np.log10(reg_df[['energy']].values)
y = np.log10(reg_df['impact-e'].values)

lr = LinearRegression().fit(X, y)
print("회귀계수(기울기):", lr.coef_[0])
print("절편:", lr.intercept_)
print("결정계수 R^2:", lr.score(X, y))

**해석**: R² = 0.9998로 거의 완벽하게 설명되는 관계이며, 이는 두 지표가 독립적인 관측값이 아니라 하나의 관측(광학 에너지)으로부터 물리 공식을 통해 유도된 값임을 시사한다.

## 7. 분석 5 — 지리적 군집 (K-Means + Elbow)

화구 관측 위치를 군집화하여, 운석 착지 데이터(6절, 남극 중심 군집)와 패턴이 다른지 확인한다.

In [ ]:
X_geo = geo[['lat_signed','lon_signed']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_geo)

inertias = []
for k in range(1,9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(6,4))
plt.plot(range(1,9), inertias, marker='o')
plt.xlabel('군집 수(k)')
plt.ylabel('Inertia')
plt.title('Elbow Method (화구 위치)')
plt.show()

In [ ]:
km = KMeans(n_clusters=4, random_state=42, n_init=10)
geo = geo.copy()
geo['cluster'] = km.fit_predict(X_scaled)

print(geo.groupby('cluster')[['lat_signed','lon_signed']].mean())
print()
print(geo['cluster'].value_counts())

plt.figure(figsize=(8,5))
sns.scatterplot(data=geo, x='lon_signed', y='lat_signed', hue='cluster', palette='tab10', s=20)
plt.title('화구 관측 위치 군집 (K-Means, k=4)')
plt.xlabel('경도')
plt.ylabel('위도')
plt.show()

### 결과 해석

운석 착지 데이터의 군집(6절)은 남극과 중위도 건조지대에 뚜렷하게 쏠려 있었던 반면,
화구 관측 데이터의 군집은 특정 위도대에 쏠리지 않고 전 지구에 비교적 고르게 분포한다.
이는 위성 관측 방식 자체에는 지리적 편향이 없음을 보여주며, 착지 데이터에서 나타난 편향이
관측 단계가 아니라 '발견·수집' 단계에서 발생한다는 해석을 강하게 뒷받침한다.

## 8. 종합 요약

| 분석 항목 | 방법 | 결과 |
|---|---|---|
| 남극권 비율 비교 | 단순 비율 계산 | 착지 데이터 68.7% vs 화구 데이터 6.57% |
| 에너지 지표 상관관계 | Pearson 상관계수 | r = 0.998 |
| 남/북반구 에너지 차이 | Shapiro-Wilk → Mann-Whitney U | p = 0.986 (유의한 차이 없음) |
| 에너지 지표 회귀분석 | 로그-로그 선형회귀 | R² = 0.9998 |
| 지리적 군집 | K-Means (k=4, Elbow) | 남극 편중 없이 전 지구에 고른 분포 |

**종합 해석**: 독립적으로 수집된 두 개의 공식 데이터셋(운석 착지 데이터, 화구 관측 데이터)을 비교한 결과, 지구 대기권에 진입하는 소천체의 실제 분포는 특정 지역에 편중되지 않는 반면, 지상에서 '발견'되어 기록된 운석은 남극에 강하게 편중되어 있다. 이는 우리가 보유한 운석 데이터가 우주의 실제 분포를 그대로 반영한 것이 아니라, 지구상의 탐사·발견 활동에 의해 상당 부분 형성된 표본임을 뒷받침한다.

**데이터 출처**: NASA/JPL CNEOS Fireball and Bolide Data (ssd-api.jpl.nasa.gov/fireball.api)